## Generate MESH_output_streamflow.csv and Basin_average_water_balance.csv files: It reads MESH_input_streamflow.tb0 and MESH Flux Output Files in NetCDF 

In [3]:
import os
import geopandas as gpd
import xarray as xr
import pandas as pd
import numpy as np
from datetime import datetime


def combine_mesh_sim_obs(
    input_stations_comids: str,
    input_obs: str,
    input_ddb: str,
    mesh_flow: str,
    csv_filename: str
):
    """
    Combine observed and simulated MESH streamflow data for stations
    and save as a CSV file.

    Optimized for:
      - Graceful handling of missing COMIDs (fills with NaN)
      - High performance DataFrame construction
      - Robust file I/O and automatic path creation
      - Clean logging and safe merges

    Parameters
    ----------
    input_stations_comids : str
        Path to the stations COMIDs GeoPackage.
    input_obs : str
        Path to the observed streamflow file (TB0 format).
    input_ddb : str
        Path to the drainage database NetCDF.
    mesh_flow : str
        Path to the simulated streamflow NetCDF.
    csv_filename : str
        Output path for the combined CSV file.
    """

    print("🔹 Reading input files...")

    # --- Read drainage database (NetCDF) ---
    with xr.open_dataset(input_ddb) as db:
        segid = db["subbasin"].values

    # --- Read station COMIDs (GeoPackage) ---
    stations_comids = gpd.read_file(input_stations_comids, layer="points")
    stations_comids["Obs_NM"] = stations_comids["Obs_NM"].astype(str)

    # Sort stations: non-USGS first, then USGS
    station_ca = sorted(stations_comids.loc[stations_comids["SRC_obs"] != "USGS", "Obs_NM"].tolist())
    station_us = sorted(stations_comids.loc[stations_comids["SRC_obs"] == "USGS", "Obs_NM"].tolist())
    StationID = station_ca + station_us
    StationCOMID = [stations_comids.loc[stations_comids["Obs_NM"] == s, "COMID"].values[0] for s in StationID]

    # --- Match COMIDs to segid ---
    print("🔹 Matching station COMIDs with segid array...")
    missing = []
    for s, b in zip(StationID, StationCOMID):
        if not np.any(segid == b):
            missing.append((s, b))

    if missing:
        print(f"⚠️ {len(missing)} COMIDs not found in segid:")
        for s, b in missing:
            print(f"   - Station {s} (COMID {b}) missing. Will fill with NaN.")
    else:
        print("✅ All station COMIDs found in segid.")

    # --- Read simulated streamflow ---
    print("🔹 Reading simulated streamflow from:", mesh_flow)
    with xr.open_dataset(mesh_flow) as meshout:
        flow = meshout["QO"].values  # Adjust variable name if needed
        model_start_time = pd.to_datetime(str(meshout.time.values[0])).normalize()

    # --- Build simulated DataFrame efficiently ---
    print("🔹 Constructing simulated streamflow DataFrame...")

    sim_data = {}
    for s, b in zip(StationID, StationCOMID):
        idx = np.where(segid == b)[0]
        if len(idx) > 0:
            sim_data[f"QOSIM_{s}"] = flow[:, idx[0]]
        else:
            sim_data[f"QOSIM_{s}"] = np.full(flow.shape[0], np.nan)

    df_sim = pd.DataFrame(sim_data)
    df_sim.insert(0, "Dates", pd.date_range(start=model_start_time, periods=len(df_sim), freq="D"))

    # --- Read observed streamflow ---
    print("🔹 Reading observed streamflow from:", input_obs)
    with open(input_obs) as f:
        start_line = next(line for line in f if line.strip().startswith(":StartTime"))
    observed_start_time = datetime.strptime(
        " ".join(start_line.split()[1:]),
        "%Y/%m/%d %H:%M:%S.%f"
    )

    df_obs = pd.read_csv(input_obs, delimiter=r"\s+", skiprows=37, header=None)
    df_obs.columns = [f"QOMEAS_{s}" for s in StationID]
    df_obs.insert(0, "Dates", pd.date_range(start=observed_start_time, periods=len(df_obs), freq="D"))

    # --- Merge observed and simulated ---
    print("🔹 Merging observed and simulated data...")
    df_list = []

    for s in StationID:
        obs_col = f"QOMEAS_{s}"
        sim_col = f"QOSIM_{s}"

        if sim_col not in df_sim.columns:
            print(f"⚠️ No simulated data for {s}, skipping merge.")
            continue

        obs = df_obs[["Dates", obs_col]].copy()
        sim = df_sim[["Dates", sim_col]].copy()

        # Align by overlapping dates
        start = max(obs["Dates"].min(), sim["Dates"].min())
        end = min(obs["Dates"].max(), sim["Dates"].max())

        obs_sub = obs[(obs["Dates"] >= start) & (obs["Dates"] <= end)]
        sim_sub = sim[(sim["Dates"] >= start) & (sim["Dates"] <= end)]

        merged = pd.merge(obs_sub, sim_sub, on="Dates", how="inner")
        if merged.empty:
            print(f"⚠️ No overlapping period for station {s}, skipping.")
            continue

        merged.insert(0, "YEAR", merged["Dates"].dt.year)
        merged.insert(1, "JDAY", merged["Dates"].dt.dayofyear)
        merged.drop(columns=["Dates"], inplace=True)

        df_list.append(merged)

    if not df_list:
        raise ValueError("❌ No stations had both observed and simulated data to merge.")

    df_combined = pd.concat(df_list, axis=1)
    df_combined = df_combined.loc[:, ~df_combined.columns.duplicated()]

    # --- Save to CSV ---
    os.makedirs(os.path.dirname(csv_filename), exist_ok=True)
    df_combined.to_csv(csv_filename, index=False)
    print(f"✅ Combined CSV saved to: {csv_filename}")

In [4]:
# Top-level folders
mesh_versions = ["Test"] # ["MESH_CaSRv2p1", "MESH_CaSRv3p1"]
gru_types = ["Average_GRU_Params", "Distributed_GRU_Params"]

# Base paths
base_path = r"D:\Zelalem\RUNs"
stations_comids_path = r"D:\Zelalem\WSC\combined_discharge_stations_comids.gpkg"
obs_path = r"D:\Zelalem\WSC\MESH_input_streamflow_latlon.tb0"
ddb_path = r"D:\Zelalem\RUNs\MESH_drainage_database_Polish_0p05_0p02_0p01.nc"

for mesh_ver in mesh_versions:
    for gru in gru_types:
        # Path to GRU folder
        gru_path = os.path.join(base_path, mesh_ver, gru)
        
        # Skip if the GRU folder does not exist
        if not os.path.exists(gru_path):
            print(f"⚠️ Skipping missing folder: {gru_path}")
            continue
        
        # Dynamically list subfolders inside this GRU folder
        mesh_subfolders = [f for f in os.listdir(gru_path) if os.path.isdir(os.path.join(gru_path, f))]
        
        for sub in mesh_subfolders:
            # Construct full paths
            mesh_flow_path = os.path.join(gru_path, sub, "QO_D_GRD.nc")
            csv_output_path = os.path.join(gru_path, sub, "MESH_output_streamflow.csv")
            
            # Skip if the QO_D_GRD.nc file does not exist
            if not os.path.exists(mesh_flow_path):
                print(f"⚠️ Skipping missing mesh flow file: {mesh_flow_path}")
                continue
            
            print(f"Running for: {mesh_ver} / {gru} / {sub}")
            print(f"Mesh flow: {mesh_flow_path}")
            print(f"CSV output: {csv_output_path}")
            
            # Call the function
            combine_mesh_sim_obs(
                input_stations_comids=stations_comids_path,
                input_obs=obs_path,
                input_ddb=ddb_path,
                mesh_flow=mesh_flow_path,
                csv_filename=csv_output_path
            )    

Running for: Test / Average_GRU_Params / MESH_1860mezt_full
Mesh flow: D:\Zelalem\RUNs\Test\Average_GRU_Params\MESH_1860mezt_full\QO_D_GRD.nc
CSV output: D:\Zelalem\RUNs\Test\Average_GRU_Params\MESH_1860mezt_full\MESH_output_streamflow.csv
🔹 Reading input files...
🔹 Matching station COMIDs with segid array...
⚠️ 1 COMIDs not found in segid:
   - Station 12448990 (COMID 78014226.0) missing. Will fill with NaN.
🔹 Reading simulated streamflow from: D:\Zelalem\RUNs\Test\Average_GRU_Params\MESH_1860mezt_full\QO_D_GRD.nc
🔹 Constructing simulated streamflow DataFrame...
🔹 Reading observed streamflow from: D:\Zelalem\WSC\MESH_input_streamflow_latlon.tb0
🔹 Merging observed and simulated data...
✅ Combined CSV saved to: D:\Zelalem\RUNs\Test\Average_GRU_Params\MESH_1860mezt_full\MESH_output_streamflow.csv
Running for: Test / Average_GRU_Params / MESH_1p5p5_full
Mesh flow: D:\Zelalem\RUNs\Test\Average_GRU_Params\MESH_1p5p5_full\QO_D_GRD.nc
CSV output: D:\Zelalem\RUNs\Test\Average_GRU_Params\MESH_1